# Audio NER with RNNs (Lab Assessment 10)
Pipeline: **voice → Whisper transcript → BIO annotation → BiRNN training → evaluation on new recordings**.

Before running: upload this project folder to Google Drive, put your 10 training recordings in `audio/train/` (`sentence_01.ogg` …) and 2–3 test recordings in `audio/test/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ner-audio-project

In [ ]:
!apt-get -qq install -y ffmpeg
!pip -q install openai-whisper SpeechRecognition

## 1. Speech-to-text
WER is computed against the reference sentences in `data/sentences.tsv` so you can quantify ASR errors.

In [ ]:
!python src/transcribe.py --audio audio/train --out data/transcripts_asr_train.tsv --reference data/sentences.tsv
!python src/transcribe.py --audio audio/test  --out data/transcripts_asr_test.tsv  --reference data/test_sentences.tsv

## 2. Annotate with BIO tags
`suggest` pre-marks entities; **review the output by hand**, copy the corrected lines into `data/sentences.tsv` / `data/test_sentences.tsv`, then `build`.

In [ ]:
!python src/annotate.py suggest --asr data/transcripts_asr_train.tsv --out data/sentences_suggested.tsv
!cat data/sentences_suggested.tsv

In [ ]:
!python src/annotate.py build --in data/sentences.tsv --prefix data/annotated_train
!python src/annotate.py build --in data/test_sentences.tsv --prefix data/annotated_test
!head -30 data/annotated_train.conll

## 3. Train SimpleRNN vs LSTM vs GRU (bidirectional)

In [ ]:
!python src/train_keras.py --cell all --epochs 30

## 4. NumPy BiGRU: ablation, evaluation, export for the website

In [ ]:
!python tests/test_gradcheck.py
!cd src && python run_experiments.py --epochs 30 --seeds 5

In [ ]:
import json
r = json.load(open('results/evaluation_results.json'))
for p in r['test_predictions']:
    print(p['id'])
    for t,g,q in zip(p['tokens'],p['gold'],p['pred']): print(f'  {t:18s} gold={g:7s} pred={q:7s}', '' if g==q else '  <-- ERROR')
print(json.dumps(r['test_cased'], indent=1))

## 5. Predict on a brand-new voice recording

In [ ]:
!python src/predict_audio.py audio/test/test_01.ogg --backend whisper